In [1]:
# Cell — Audit: are weekly totals contaminated by store closures?
#
# Rossmann records `Sales = 0` on days when `Open = 0` (Sundays, public
# holidays, refurbishment). Summing daily sales into weekly totals therefore
# conflates two different things: a drop in demand, and a drop in trading days.
# A model trained on the raw weekly sum learns the trading calendar and
# mislabels it as seasonality, inflating apparent demand volatility and hence
# safety stock.

import pandas as pd

raw = pd.read_csv("../data/raw/train.csv", parse_dates=["Date"], low_memory=False)
print("Columns:", list(raw.columns))

stores = [733, 198]
d = raw[raw["Store"].isin(stores)].copy()
d["Week"] = d["Date"].dt.to_period("W").dt.end_time.dt.normalize()

audit = (
    d.groupby(["Store", "Week"])
     .agg(
         sales_sum=("Sales", "sum"),
         open_days=("Open", "sum"),
         days_recorded=("Open", "size"),
         promo_days=("Promo", "sum"),
     )
     .reset_index()
)

print("\nDistribution of open days per week:")
print(audit.groupby(["Store", "open_days"]).size().unstack(fill_value=0))

print("\nWeeks with fewer than 6 open days:")
odd = audit[audit["open_days"] < 6]
print(f"  {len(odd)} of {len(audit)} store-weeks ({len(odd)/len(audit):.1%})")
print(odd.sort_values("open_days").head(12).to_string(index=False))

print("\nAny gap in the calendar (store closed for a full week)?")
print(audit[audit["days_recorded"] < 7].to_string(index=False))

Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

Distribution of open days per week:
open_days  4   5    6    7
Store                     
198        3  22  110    0
733        0   1    1  133

Weeks with fewer than 6 open days:
  26 of 270 store-weeks (9.6%)
 Store       Week  sales_sum  open_days  days_recorded  promo_days
   198 2013-01-06       7553          4              6           0
   198 2014-12-28      13510          4              7           0
   198 2013-12-29      11312          4              7           0
   198 2015-05-31      10445          5              7           0
   198 2015-05-17      10100          5              7           0
   198 2015-05-03      20278          5              7           5
   198 2015-04-12      10423          5              7           0
   198 2015-04-05      21628          5              7           5
   198 2015-01-04       9773          5              7           0
   198

In [2]:
# Cell — Quantify how much of the observed volatility is calendar noise
#
# Weekly sales decompose into an intensity term and a calendar term:
#     weekly_sales = sales_per_trading_day * trading_days
# The calendar term is known in advance (German public holidays are published
# years ahead), so it does not require a forecast and does not require safety
# stock. Only the intensity term is genuinely uncertain. This cell measures how
# much of each store's apparent volatility is calendar rather than demand.

# Drop the first and last weeks: they are truncated by the dataset boundary,
# not by a closure (days_recorded < 7).
full = audit[audit["days_recorded"] == 7].copy()
print(f"Complete weeks retained: {len(full)} of {len(audit)}")

full["sales_per_day"] = full["sales_sum"] / full["open_days"]

comparison = full.groupby("Store").agg(
    weeks=("Week", "count"),
    raw_mean=("sales_sum", "mean"),
    raw_std=("sales_sum", "std"),
    rate_mean=("sales_per_day", "mean"),
    rate_std=("sales_per_day", "std"),
)
comparison["CV_raw"] = comparison["raw_std"] / comparison["raw_mean"]
comparison["CV_adjusted"] = comparison["rate_std"] / comparison["rate_mean"]
comparison["CV_reduction_%"] = (
    (comparison["CV_raw"] - comparison["CV_adjusted"]) / comparison["CV_raw"] * 100
)

print("\nVolatility before and after removing the trading-day effect:")
print(comparison[["weeks", "CV_raw", "CV_adjusted", "CV_reduction_%"]].round(3))

# Trading days are not interchangeable: a lost Saturday costs more than a lost
# Monday. Check how uneven the weekday profile is before assuming they are.
dow = (
    d[d["Open"] == 1]
    .groupby(["Store", "DayOfWeek"])["Sales"]
    .mean()
    .unstack()
)
print("\nMean sales by day of week (1=Mon ... 7=Sun):")
print(dow.round(0))

Complete weeks retained: 266 of 270

Volatility before and after removing the trading-day effect:
       weeks  CV_raw  CV_adjusted  CV_reduction_%
Store                                            
198      133   0.362        0.349           3.463
733      133   0.087        0.087           0.000

Mean sales by day of week (1=Mon ... 7=Sun):
DayOfWeek        1        2        3        4        5        6        7
Store                                                                   
198         4154.0   3756.0   2719.0   2852.0   3044.0    936.0      NaN
733        15542.0  14562.0  14488.0  14815.0  15966.0  14015.0  15144.0


In [3]:
# Cell — Investigate the Saturday anomaly at store 198
#
# Store 198 averages 936 EUR on Saturdays against ~3,000-4,000 on weekdays.
# In retail, Saturday is normally the strongest day, so this warrants a check
# before the weekly aggregates are trusted. Rossmann is known to contain rows
# with Open = 1 and Sales = 0; if these cluster on Saturdays they would deflate
# both the daily mean and every weekly total built from it.

s198 = d[(d["Store"] == 198) & (d["Open"] == 1)]

print("Open days recording zero sales, by day of week:")
zero = s198[s198["Sales"] == 0].groupby("DayOfWeek").size()
total = s198.groupby("DayOfWeek").size()
print(pd.DataFrame({"open_days": total, "zero_sales": zero.reindex(total.index, fill_value=0)}))

print("\nSaturday sales distribution (store 198, open days only):")
sat = s198[s198["DayOfWeek"] == 6]["Sales"]
print(sat.describe().round(0))

print("\nSaturday sales excluding zeros:")
print(sat[sat > 0].describe().round(0))

print("\nSame check for store 733:")
s733 = d[(d["Store"] == 733) & (d["Open"] == 1)]
print(s733.groupby("DayOfWeek")["Sales"].agg(["count", "mean", "median"]).round(0))

Open days recording zero sales, by day of week:
           open_days  zero_sales
DayOfWeek                       
1                128           0
2                134           0
3                132           0
4                126           0
5                128           0
6                134           0

Saturday sales distribution (store 198, open days only):
count     134.0
mean      936.0
std       327.0
min       530.0
25%       747.0
50%       866.0
75%      1030.0
max      3223.0
Name: Sales, dtype: float64

Saturday sales excluding zeros:
count     134.0
mean      936.0
std       327.0
min       530.0
25%       747.0
50%       866.0
75%      1030.0
max      3223.0
Name: Sales, dtype: float64

Same check for store 733:
           count     mean   median
DayOfWeek                         
1            134  15542.0  15386.0
2            135  14562.0  14495.0
3            135  14488.0  14178.0
4            135  14815.0  14400.0
5            135  15966.0  15852.0
6            

In [5]:
# Cell — Does promotion explain store 198's volatility?
#
# Promotion weeks in the diagnostic above sold roughly twice as much as
# non-promotion weeks with the same number of trading days. If promotion is the
# dominant driver, then most of store 198's apparent uncertainty is not
# uncertainty at all: the retailer schedules its own promotions and knows the
# calendar in advance. Safety stock covers what cannot be anticipated, so a
# promo-aware forecast should reduce required stock without reducing service.

full = audit[audit["days_recorded"] == 7].copy()
full["is_promo"] = full["promo_days"] > 0

print("Weekly sales split by promotion status:")
split = full.groupby(["Store", "is_promo"])["sales_sum"].agg(
    ["count", "mean", "std"]
).round(0)
split["CV"] = (split["std"] / split["mean"]).round(3)
print(split)

print("\nOverall CV vs CV within each promotion group:")
for store, g in full.groupby("Store"):
    overall = g["sales_sum"].std() / g["sales_sum"].mean()
    within = g.groupby("is_promo")["sales_sum"].apply(lambda x: x.std() / x.mean())
    print(f"  Store {store}: overall {overall:.3f} | "
          f"no promo {within.get(False, float('nan')):.3f} | "
          f"promo {within.get(True, float('nan')):.3f}")

print("\nPromo weeks per store:")
print(full.groupby(["Store", "promo_days"]).size().unstack(fill_value=0))

Weekly sales split by promotion status:
                count      mean     std     CV
Store is_promo                                
198   False        62   10884.0  1206.0  0.111
      True         71   22062.0  3135.0  0.142
733   False        62  100448.0  6873.0  0.068
      True         71  108365.0  9259.0  0.085

Overall CV vs CV within each promotion group:
  Store 198: overall 0.362 | no promo 0.111 | promo 0.142
  Store 733: overall 0.087 | no promo 0.068 | promo 0.085

Promo weeks per store:
promo_days   0   5
Store             
198         62  71
733         62  71


In [6]:
# Cell — Variance decomposition: how much of each store's volatility is
# explained by the promotion calendar?
#
# Promotion is scheduled by the retailer and known weeks in advance. Any
# variance it explains is anticipated variation, not uncertainty, and therefore
# should not be absorbed by safety stock. This computes eta-squared: the share
# of total weekly variance attributable to promotion status.

full = audit[audit["days_recorded"] == 7].copy()
full["is_promo"] = full["promo_days"] > 0

for store, g in full.groupby("Store"):
    grand_mean = g["sales_sum"].mean()
    ss_total = ((g["sales_sum"] - grand_mean) ** 2).sum()
    ss_between = sum(
        len(sub) * (sub["sales_sum"].mean() - grand_mean) ** 2
        for _, sub in g.groupby("is_promo")
    )
    eta_sq = ss_between / ss_total

    cv_total = g["sales_sum"].std() / grand_mean
    residual = g["sales_sum"] - g.groupby("is_promo")["sales_sum"].transform("mean")
    cv_residual = residual.std() / grand_mean

    print(
        f"Store {store}: promotion explains {eta_sq:6.1%} of weekly variance | "
        f"CV {cv_total:.3f} -> {cv_residual:.3f} "
        f"({(cv_total - cv_residual) / cv_total:.0%} reduction)"
    )

Store 198: promotion explains  84.2% of weekly variance | CV 0.362 -> 0.144 (60% reduction)
Store 733: promotion explains  18.9% of weekly variance | CV 0.087 -> 0.078 (10% reduction)
